# Train a proprioception-tuned Multi-View Vision Transformer (ViT)
We create a sensor processing model using multiple Vision Transformer (ViT) based visual encoders
finetuned with proprioception.
We start with pretrained ViT models, then train them to:
1. Create a meaningful 128-dimensional latent representation from multiple camera views
2. Learn to map this representation to robot positions (proprioception)
 The sensor processing object associated with the trained model is in sensorprocessing/sp_vit_multiview.py


In [ ]:
import sys
sys.path.append("..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
import torch
import torch.nn as nn
from torchvision import models, transforms
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

from demonstration.demonstration import Demonstration
from sensorprocessing.helper_training_data import load_multiview_images_as_proprioception_training
from sensorprocessing.sp_vit_multiview import MultiViewVitSensorProcessing

In [ ]:

# The experiment/run we are going to run: the specified model will be created
experiment = "sensorprocessing_propriotuned_Vit_multiview"


# Other possible configurations:

#concat_proj
# run = "vit_base_multiview"  # ViT Base
run = "vit_large_multiview_128" # ViT Large
# run = "vit_huge_multiview" # ViT Huge

##  indiv_proj
# run = "vit_base_multiview_indiv_proj"  # ViT Base_indiv_proj
# run = "vit_large_multiview_indiv_proj" # ViT Large_indiv_proj
# run = "vit_huge_multiview_indiv_proj" # ViT Huge


##  attention
# run = "vit_base_multiview_attention"  # ViT Base_attention
# run = "vit_large_multiview_attention" # ViT Large_attention
# run = "vit_huge_multiview_attention" # ViT Huge_attention

##  weighted_sum
# run = "vit_base_multiview_weighted_sum"  # ViT Base_weighted_sum
# run = "vit_large_multiview_weighted_sum" # ViT Large_weighted_sum
# run = "vit_huge_multiview_weighted_sum" # ViT Huge_weighted_sum

##  gated
# run = "vit_base_multiview_gated"  # ViT Base_gated
# run = "vit_large_multiview_gated" # ViT Large_gated
# run = "vit_huge_multiview_gated" # ViT Huge_gated

exp = Config().get_experiment(experiment, run)


In [ ]:
# ### Exp-run initialization for papermill automation

# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"

experiment = "sensorprocessing_propriotuned_Vit_multiview"
run = "vit_large_multiview_128"
epochs = None

# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

In [ ]:
# Create output directory if it doesn't exist
data_dir = pathlib.Path(exp["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)
print(f"Data directory: {data_dir}")



In [ ]:
# Handle external paths (when called from Flow)
if expruns_path:
    expruns_path = pathlib.Path(expruns_path).expanduser()
    expruns_path.mkdir(parents=True, exist_ok=True)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment("sensorprocessing_propriotuned_Vit_multiview")
    Config().copy_experiment("robot_al5d")
    Config().copy_experiment("demonstration")

if results_path:
    results_path = pathlib.Path(results_path).expanduser()
    results_path.mkdir(parents=True, exist_ok=True)
    Config().set_results_path(results_path)

# The experiment/run we are going to run
exp = Config().get_experiment(experiment, run, creation_style=creation_style)
exp_robot = Config().get_experiment(exp["robot_exp"], exp["robot_run"])

### Create regression training data (image to proprioception)
The training data (XX, Y) is all the 2-view pictures from a demonstration with the corresponding proprioception data.

In [ ]:
# Multiview data preparation is shared through helper_training_data.

In [ ]:
# The legacy notebook-local loader was removed.

In [ ]:
modelfile = pathlib.Path(exp["data_dir"], exp["proprioception_mlp_model_file"])

if modelfile.exists():
    print("*** Train-ProprioTuned-ViT-Multiview ***: NOT training; model already exists")

tr = load_multiview_images_as_proprioception_training(exp, exp_robot)
view_inputs_training = tr["view_inputs_training"]
targets_training = tr["targets_training"]
view_inputs_validation = tr["view_inputs_validation"]
targets_validation = tr["targets_validation"]

### Create the multi-view ViT model


In [ ]:

# Create the multi-view ViT model
sp = MultiViewVitSensorProcessing(exp)
model = sp.enc  # Get the actual encoder model for training

print("Model created successfully")

try:
    params = model.parameters()
    print("Parameters accessed successfully")
    param_count = sum(p.numel() for p in params)
    print(f"Total parameters: {param_count}")
except Exception as e:
    print(f"Error accessing parameters: {e}")

# Select loss function
loss_type = exp.get('loss', 'MSELoss')
if loss_type == 'MSELoss':
    criterion = nn.MSELoss()
elif loss_type == 'L1Loss':
    criterion = nn.L1Loss()
else:
    criterion = nn.MSELoss()  # Default to MSE

# Set up optimizer with appropriate learning rate and weight decay
optimizer = optim.Adam(
    model.parameters(),
    lr=exp.get('learning_rate', 0.001),
    weight_decay=exp.get('weight_decay', 0.01)
)

# Optional learning rate scheduler
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)


### Custom dataset for multi-view data

In [ ]:

class MultiViewDataset(torch.utils.data.Dataset):
    def __init__(self, view_inputs, targets):
        self.view_inputs = view_inputs  # List of tensors, one per view
        self.targets = targets
        self.num_samples = len(targets)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Get corresponding sample from each view
        views = [view[idx] for view in self.view_inputs]
        target = self.targets[idx]
        return views, target

In [ ]:

# Create DataLoaders for batching
batch_size = exp.get('batch_size', 32)
train_dataset = MultiViewDataset(view_inputs_training, targets_training)
test_dataset = MultiViewDataset(view_inputs_validation, targets_validation)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train_and_save_multiview_proprioception_model(model, criterion, optimizer, modelfile,
                                                epochs=20, scheduler=None,
                                                log_interval=1):
    """Trains and saves the multi-view ViT proprioception model with checkpoint support

    Args:
        model: Multi-view ViT model with proprioception
        criterion: Loss function
        optimizer: Optimizer
        modelfile: Path to save the final model
        epochs: Number of training epochs
        scheduler: Optional learning rate scheduler
        log_interval: How often to print logs
    """
    # Create checkpoint directory
    checkpoint_dir = modelfile.parent / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)

    # Maximum number of checkpoints to keep (excluding the best model)
    max_checkpoints = 2

    # Ensure model is on the right device
    model = model.to(Config().runtime["device"])
    criterion = criterion.to(Config().runtime["device"])

    # Function to extract epoch number from checkpoint file
    def get_epoch_number(checkpoint_file):
        try:
            # Use a more robust approach to extract epoch number
            # Format: epoch_XXXX.pth where XXXX is the epoch number
            filename = checkpoint_file.stem
            parts = filename.split('_')
            if len(parts) >= 2:
                return int(parts[1])  # Get the number after "epoch_"
            return 0
        except:
            return 0

    # Keep track of the best validation loss
    best_val_loss = float('inf')
    start_epoch = 0

    # Check for existing checkpoints to resume from
    checkpoint_files = list(checkpoint_dir.glob("epoch_*.pth"))
    if checkpoint_files:
        # Sort by epoch number for more reliable ordering
        checkpoint_files.sort(key=get_epoch_number)

        # Get the most recent checkpoint
        latest_checkpoint = checkpoint_files[-1]
        epoch_num = get_epoch_number(latest_checkpoint)

        print(f"Found checkpoint from epoch {epoch_num}. Resuming training...")

        # Load checkpoint
        checkpoint = torch.load(latest_checkpoint, map_location=Config().runtime["device"])
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint.get('best_val_loss', float('inf'))

        print(f"Resuming from epoch {start_epoch}/{epochs} with best validation loss: {best_val_loss:.4f}")

    # Function to clean up old checkpoints
    def cleanup_old_checkpoints():
        # Get all epoch checkpoint files
        checkpoint_files = list(checkpoint_dir.glob("epoch_*.pth"))

        # Sort by actual epoch number, not just filename
        checkpoint_files.sort(key=get_epoch_number)

        if len(checkpoint_files) > max_checkpoints:
            files_to_delete = checkpoint_files[:-max_checkpoints]
            for file_path in files_to_delete:
                try:
                    file_path.unlink()
                    print(f"Deleted old checkpoint: {file_path.name}")
                except Exception as e:
                    print(f"Failed to delete {file_path.name}: {e}")

    # Training loop
    for epoch in range(start_epoch, epochs):
        print(f"Starting epoch {epoch+1}/{epochs}")

        # Training phase
        model.train()
        total_loss = 0
        for batch_idx, (batch_views, batch_y) in enumerate(train_loader):
            # Move views and targets to device
            batch_views = [view.to(Config().runtime["device"]) for view in batch_views]
            batch_y = batch_y.to(Config().runtime["device"])

            # Forward pass through the full model (including proprioceptor)
            predictions = model.forward(batch_views)
            loss = criterion(predictions, batch_y)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # Print progress every few batches
            if (batch_idx + 1) % 10 == 0:
                print(f"  Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")

        avg_train_loss = total_loss / len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_views, batch_y in test_loader:
                # Move views and targets to device
                batch_views = [view.to(Config().runtime["device"]) for view in batch_views]
                batch_y = batch_y.to(Config().runtime["device"])

                predictions = model(batch_views)
                loss = criterion(predictions, batch_y)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(test_loader)

        # Update learning rate if scheduler is provided
        if scheduler is not None:
            scheduler.step(avg_val_loss)

        # Save checkpoint for this epoch - using formatted epoch numbers for reliable sorting
        checkpoint_path = checkpoint_dir / f"epoch_{epoch:06d}.pth"
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'best_val_loss': best_val_loss
        }, checkpoint_path)
        print(f"Checkpoint saved to {checkpoint_path}")

        # Clean up old checkpoints to save space
        cleanup_old_checkpoints()

        # Save the best model separately
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_path = checkpoint_dir / "best_model.pth"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train_loss,
                'val_loss': avg_val_loss,
                'best_val_loss': best_val_loss
            }, best_model_path)
            print(f"  New best model saved with validation loss: {best_val_loss:.4f}")

        # Log progress
        if (epoch + 1) % log_interval == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

    # Training completed successfully
    print(f"Training complete. Best validation loss: {best_val_loss:.4f}")

    # Load the best model for final save
    best_model_path = checkpoint_dir / "best_model.pth"
    if best_model_path.exists():
        best_checkpoint = torch.load(best_model_path, map_location=Config().runtime["device"])
        model.load_state_dict(best_checkpoint['model_state_dict'])
        print(f"Loaded best model from epoch {best_checkpoint['epoch']+1} with loss {best_checkpoint['val_loss']:.4f}")

    # Save to final model file only after completing all epochs
    torch.save(model.state_dict(), modelfile)
    print(f"Final model saved to {modelfile}")

    return model


# Set up model file path and epochs
modelfile = pathlib.Path(exp["data_dir"], exp["proprioception_mlp_model_file"])
epochs = exp.get("epochs", 20)

# First check for existing final model
if modelfile.exists() and exp.get("reload_existing_model", True):
    print(f"Loading existing final model from {modelfile}")
    model.load_state_dict(torch.load(modelfile, map_location=Config().runtime["device"]))

    # Evaluate the loaded model
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for batch_views, batch_y in test_loader:
            # Move views and targets to device
            batch_views = [view.to(Config().runtime["device"]) for view in batch_views]
            batch_y = batch_y.to(Config().runtime["device"])

            predictions = model(batch_views)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()

        avg_val_loss = val_loss / len(test_loader)
        print(f"Loaded model validation loss: {avg_val_loss:.4f}")
else:
    # Check for checkpoints to resume from, otherwise start fresh training
    checkpoint_dir = modelfile.parent / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)

    # Use the get_epoch_number function for reliable sorting
    def get_epoch_number(checkpoint_file):
        try:
            filename = checkpoint_file.stem
            parts = filename.split('_')
            if len(parts) >= 2:
                return int(parts[1])
            return 0
        except:
            return 0

    checkpoint_files = list(checkpoint_dir.glob("epoch_*.pth"))
    if checkpoint_files:
        # Sort by epoch number
        checkpoint_files.sort(key=get_epoch_number)
        latest_epoch = get_epoch_number(checkpoint_files[-1])
        print(f"Found checkpoints up to epoch {latest_epoch}. Will resume training from last checkpoint.")
    else:
        print(f"Starting new training for {epochs} epochs")

    # Train the model with checkpoint support
    model = train_and_save_multiview_proprioception_model(
        model, criterion, optimizer, modelfile,
 epochs=epochs, scheduler=lr_scheduler
    )

### Test the trained model


In [ ]:

# Create the sensor processing module using the trained model
sp = MultiViewVitSensorProcessing(exp)

# Test the sensor processor through the Demonstration-backed API.
def test_multiview_sensor_processing(sp, exp, dataset_name="validation_data", n_samples=5):
    """Test multiview inference using the configured Demonstration."""
    dataset = exp.get(dataset_name) or exp["training_data"]
    run, demo_name, cameras = dataset[0]
    demo_exp = Config().get_experiment("demonstration", run)
    demo = Demonstration(demo_exp, demo_name)

    print(f"\nTesting multiview sensor processing with {demo_name}:")
    print("-" * 60)
    for timestep in range(min(n_samples, demo.metadata["maxsteps"])):
        latent = sp.process_demonstration(demo, timestep, cameras)
        assert latent.shape[-1] == exp["latent_size"]
        print(f"  Timestep {timestep}: latent shape {latent.shape}")

test_multiview_sensor_processing(sp, exp)

### Verify the model's encoding and forward methods

In [ ]:

model.eval()
with torch.no_grad():
    # Get a sample set of views
    sample_views = [view[0].unsqueeze(0).to(Config().runtime["device"]) for view in view_inputs_validation]

    # Get the latent representation using encode
    latent = model.encode(sample_views)
    print(f"Latent representation shape: {latent.shape}")

    # Get the robot position prediction using forward
    position = model.forward(sample_views)
    print(f"Robot position prediction shape: {position.shape}")

    # Check that the latent representation has the expected size
    expected_latent_size = exp["latent_size"]
    assert latent.shape[1] == expected_latent_size, f"Expected latent size {expected_latent_size}, got {latent.shape[1]}"

    # Check that the position prediction has the expected size
    expected_output_size = exp["output_size"]
    assert position.shape[1] == expected_output_size, f"Expected output size {expected_output_size}, got {position.shape[1]}"

    print("Verification successful!")


### Test Demonstration-backed multiview inference

In [ ]:
# The complete, ordered view set is obtained from Demonstration rather than files.
test_multiview_sensor_processing(sp, exp, n_samples=3)

### Save final model and print summary

In [ ]:

final_modelfile = pathlib.Path(exp["data_dir"], exp["proprioception_mlp_model_file"])
torch.save(model.state_dict(), final_modelfile)
print(f"Model saved to {final_modelfile}")

print("\nTraining complete!")
print(f"Vision Transformer type: {exp['vit_model']}")
print(f"Number of views: {exp.get('num_views', 2)}")
print(f"Fusion type: {exp.get('fusion_type', 'concat_proj')}")
print(f"Latent space dimension: {exp['latent_size']}")
print(f"Output dimension (robot DOF): {exp['output_size']}")
print(f"Use the MultiViewVitSensorProcessing class to load and use this model for inference.")